# CELL 1 -> 07 Evaluation Metrics
Objective:

Compare Custom Decoder Transformer and LLM based generation.

Metrics:

1. BLEU Score
2. ROUGE Score
3. Slot Preservation Accuracy
4. Exact Match
5. Latency Comparison

Evaluation focuses on:

- Language quality
- Slot correctness
- Hallucination tendency

# CELL 2: Install Required Packages

pip install evaluate rouge-score nltk
pip freeze > requirements.txt


In [1]:
# CELL 3 - Import 
import json
import re
import time

import pandas as pd

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

In [2]:
# CELL 4 - Load Both Output Files

with open(
    "../outputs/decoder_predictions.json"
) as f:

    decoder_predictions = json.load(f)


with open(
    "../outputs/llm_predictions.json"
) as f:

    llm_predictions = json.load(f)



print(
    "Decoder:",
    len(decoder_predictions)
)


print(
    "LLM:",
    len(llm_predictions)
)

Decoder: 60
LLM: 60


In [ ]:
# CELL 5 — BLEU Score Function

# Full form:

# BLEU — Bilingual Evaluation Understudy

# Purpose:

# Measures overlap between generated and expected text.

def calculate_bleu(reference, generated):


    smoothing = (
        SmoothingFunction()
        .method1
    )


    score = sentence_bleu(

        [
            reference.split()
        ],

        generated.split(),

        smoothing_function=smoothing

    )


    return score

# Test the function with a simple example:
calculate_bleu(
    "I can help with order status",
    "I can help with order status"
)

1.0

In [7]:
# CELL 6 — Slot Accuracy Function

# Important metric for PS2.

# Checks if IDs are preserved.

def extract_ids(text):


    pattern = r"[A-Z]{3}[0-9]+"

    return re.findall(
        pattern,
        text
    )




def slot_accuracy(expected, generated):


    expected_ids = extract_ids(
        expected
    )


    generated_ids = extract_ids(
        generated
    )


    if len(expected_ids) == 0:

        return 1


    for value in expected_ids:


        if value not in generated_ids:

            return 0


    return 1

# Test the function with a simple example:
slot_accuracy(
    "Order ID: ABC123",
    "Order ID: ABC123"
)

1

In [8]:
# CELL 7 — Evaluate Both Models

def evaluate_model(predictions):


    bleu_scores = []

    slot_scores = []


    for item in predictions:


        bleu_scores.append(

            calculate_bleu(

                item["expected"],

                item["generated"]

            )

        )


        slot_scores.append(

            slot_accuracy(

                item["expected"],

                item["generated"]

            )

        )



    return {

        "BLEU":

        sum(bleu_scores)
        /
        len(bleu_scores),


        "Slot Accuracy":

        sum(slot_scores)
        /
        len(slot_scores)

    }

In [ ]:
# CELL 8 — Compare Results


decoder_result = evaluate_model(
    decoder_predictions
)


llm_result = evaluate_model(
    llm_predictions
)


decoder_result["Exact Match"] = exact_match(
    decoder_predictions
)


llm_result["Exact Match"] = exact_match(
    llm_predictions
)

results = pd.DataFrame(

    [

        {
            "Model":
            "Decoder Transformer",

            **decoder_result
        },


        {
            "Model":
            "LLM FLAN-T5",

            **llm_result
        }

    ]

)


results

,Model,BLEU,Slot Accuracy,Exact Match
0,Decoder Transformer,0.597315,0.266667,0.066667
1,LLM FLAN-T5,0.110107,0.866667,0.000000


In [12]:
# CELL 9 - Save Evaluation

results.to_csv(
    "../outputs/evaluation_results.csv",
    index=False
)


print(
    "Evaluation saved"
)

Evaluation saved


In [10]:
# CELL 10 - Exact Match

def exact_match(predictions):


    count = 0


    for item in predictions:


        if (
            item["expected"].lower().strip()
            ==
            item["generated"].lower().strip()
        ):

            count += 1


    return count / len(predictions)



print(
    "Decoder Exact Match:",
    exact_match(decoder_predictions)
)


print(
    "LLM Exact Match:",
    exact_match(llm_predictions)
)

Decoder Exact Match: 0.06666666666666667
LLM Exact Match: 0.0


# Final Evaluation Summary


| Metric | Decoder Transformer | FLAN-T5 LLM | Observation |
|-|-|-|-|
| BLEU | 0.597 | 0.110 | Decoder follows training templates better |
| Slot Accuracy | 0.267 | 0.867 | LLM preserves dynamic slot values better |
| Exact Match | 0.067 | 0.000 | Both generate varied responses |

## Key Observation

The custom decoder achieved better lexical similarity because it learned dataset-specific response structures.

However, the LLM achieved significantly better slot preservation, which is critical for task-oriented dialogue systems.


## Final Recommendation

For production conversational AI:

- Use LLMs for flexible response generation.
- Add slot validation layer to avoid hallucination.
- Use smaller decoder models where cost and latency are more important.